<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/07_calderon_preguntas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 07 · Preguntas sobre el teatro de Calderón: modelo base, RAG y ajuste fino

**Objetivo:** hacer preguntas sobre obras de Calderón y comparar cuatro estrategias:

| | Estrategia | Qué hace |
|---|---|---|
| A | **Modelo base** | Responde con lo que «recuerda» de su entrenamiento general. |
| B | **RAG** (*Retrieval-Augmented Generation*) | Primero **busca** versos relevantes en el corpus y se los da al modelo junto con la pregunta. |
| C | **SFT** (ajuste fino con LoRA) | Entrenamos el modelo con ~2000 preguntas y respuestas generadas automáticamente a partir del TEI. |
| D | **SFT + RAG** | El modelo ajustado recibe además los fragmentos encontrados. |

**Corpus:** 12 obras en TEI-XML (8 comedias/dramas de EMOTHE/ARTELOPE, 3 autos sacramentales en CC0 y
*Los empeños de un acaso* en la edición de Simon Kroll). Fuentes y licencias: `data/FUENTES.md`.

⏱️ Con GPU T4: unos 15–20 min en total (el entrenamiento, ~10 min). Sin GPU o sin tiempo: usa la
**copia de seguridad** ya entrenada (`ENTRENAR = False`).

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config, set_seed
cfg = load_config(overrides={
    "calderon": {
        # "epochs": 1,              # menos tiempo de entrenamiento
        # "max_train_pairs": 1000,
    }
})
ajustes = cfg.calderon
set_seed(cfg.seed)

ENTRENAR = True        # False = usar el adaptador ya entrenado de backups/ (plan B)

# Preguntas de prueba: (pregunta, respuesta esperada según el corpus)
PREGUNTAS = [
    ("¿Quién es Cosme en «La dama duende»?", "El gracioso (criado de don Manuel)."),
    ("¿Qué personajes hablan más en «La vida es sueño»?", "Segismundo, Rosaura y Clotaldo."),
    ("¿Quién dice estos versos en «La vida es sueño»?: «Hipogrifo violento, / que corriste parejas con el viento»",
     "Rosaura, al comienzo de la jornada 1 (¡en esta edición: «Hipogrifo triste»!)."),
    ("¿Quién es Pedro Crespo en «El alcalde de Zalamea»?", "Un labrador (rico) de Zalamea, padre de Isabel."),
    ("¿Cuántas jornadas tiene «El médico de su honra»?", "Tres."),
    ("¿Por qué está encerrado Segismundo en una torre?",
     "Su padre, el rey Basilio, lo encierra por un horóscopo que anunciaba que sería un rey tirano."),
]

## 1. Leer las obras (TEI-XML)
El TEI marca jornadas, personajes (`<sp>`, `<speaker>`), versos (`<l>`) y acotaciones (`<stage>`).

In [ ]:
from dhllm import calderon as cal
obras = cal.load_plays(ajustes.tei_folder)

In [ ]:
obra = obras[0]
print(cal.play_card(obra), "\n")
print(cal.play_to_text(obra)[:1200])

## 2. Cargar el modelo

In [ ]:
from dhllm.finetune import load_base_model
modelo, tokenizer = load_base_model(ajustes.base_model)

## 3. (A) Preguntar al modelo base
Fíjate en si las respuestas son **correctas** o solo **suenan bien**. Un modelo de lenguaje siempre
produce un texto plausible, aunque no «sepa» la respuesta: eso es una **alucinación**.

In [ ]:
from dhllm.sft import ask
resultados = {p: {"pregunta": p, "esperada": e} for p, e in PREGUNTAS}
for p, _ in PREGUNTAS:
    resultados[p]["A · base"] = ask(modelo, tokenizer, p, system=cal.SYSTEM_CALDERON, max_new_tokens=60, temperature=0)
    print(f"❓ {p}\n💬 {resultados[p]['A · base']}\n")

## 4. (B) RAG: buscar primero, responder después
Construimos un **índice de búsqueda** (BM25, la técnica clásica de los buscadores) con fragmentos de
~12 versos y una «ficha» por obra (reparto, jornadas...). Para cada pregunta buscamos los fragmentos más
parecidos y se los damos al modelo.

In [ ]:
from pathlib import Path
indice = cal.BM25Index(cal.make_fragments(obras))
print(len(indice.fragments), "fragmentos en el índice")
# Plan B: índice precalculado → indice = cal.BM25Index.load(ajustes.backup_index)

Así ve el buscador una pregunta:

In [ ]:
cal.show_fragments(indice.search(PREGUNTAS[0][0], k=ajustes.rag_k))

In [ ]:
for p, _ in PREGUNTAS:
    respuesta, _frags = cal.ask_with_rag(modelo, tokenizer, p, indice, k=ajustes.rag_k, max_new_tokens=80)
    resultados[p]["B · RAG"] = respuesta
    print(f"❓ {p}\n💬 {respuesta}\n")

## 5. (C) Ajuste fino (SFT) con preguntas generadas automáticamente
A partir del TEI generamos preguntas con respuesta **verificable**: reparto, papel de cada personaje,
quién habla más, quién dice unos versos, de qué obra son, cómo continúa un pasaje... Algunas llevan
fragmentos de RAG para que el modelo aprenda a usarlos. Reservamos unos pares para evaluar.

In [ ]:
pares_train, pares_test = cal.prepare_training_pairs(obras, indice, ajustes, seed=cfg.seed)
pares_train[:3]

In [ ]:
from dhllm.finetune import prepare_model, load_adapter, save_adapter
from dhllm.sft import train_sft

if ENTRENAR:
    modelo = prepare_model(modelo, ajustes.method, ajustes.lora_r, ajustes.lora_alpha)
    historial = train_sft(modelo, tokenizer, pares_train, ajustes)
    save_adapter(modelo, ajustes.save_dir)
else:
    modelo = load_adapter(modelo, ajustes.backup_adapter)

## 6. (C) y (D): preguntar al modelo ajustado, sin y con RAG

In [ ]:
for p, _ in PREGUNTAS:
    resultados[p]["C · SFT"] = ask(modelo, tokenizer, p, system=cal.SYSTEM_CALDERON, max_new_tokens=60, temperature=0)
    resultados[p]["D · SFT+RAG"] = cal.ask_with_rag(modelo, tokenizer, p, indice, k=ajustes.rag_k, max_new_tokens=80)[0]

## 7. Comparación
Lee la tabla con ojos de filólogo/a: ¿qué columna acierta?, ¿cuál inventa?, ¿cuál cita?

In [ ]:
cal.comparison_table(list(resultados.values()))

### ¿Y con preguntas que el modelo **no** vio al entrenar?
Estos pares se generaron igual que los de entrenamiento, pero los reservamos. Comparamos el modelo
ajustado sin y con RAG (con `modelo.disable_adapter()` podemos ver también el modelo base).

In [ ]:
filas = []
for par in pares_test[:6]:
    with modelo.disable_adapter():
        base = ask(modelo, tokenizer, par["pregunta"], system=cal.SYSTEM_CALDERON, max_new_tokens=50, temperature=0)
    filas.append({
        "pregunta": par["pregunta"], "esperada": par["respuesta"], "A · base": base,
        "C · SFT": ask(modelo, tokenizer, par["pregunta"], system=cal.SYSTEM_CALDERON, max_new_tokens=50, temperature=0),
        "D · SFT+RAG": cal.ask_with_rag(modelo, tokenizer, par["pregunta"], indice, k=ajustes.rag_k, max_new_tokens=60)[0],
    })
cal.comparison_table(filas)

## 💬 Para discutir: *el ajuste fino cambia el comportamiento y el estilo; el RAG aporta los hechos*

- El **modelo base** inventa con seguridad: nombres, tramas, fechas.
- El **RAG** encuentra los versos reales; el modelo los resume y puede **citar** el fragmento. Si el buscador
  no encuentra el pasaje correcto (pregunta sobre la *trama*, p. ej. Segismundo en la torre), el modelo
  tampoco acierta: la calidad depende de la búsqueda.
- El **SFT** enseña el **formato** («Los dice X, en la jornada N») y algunos datos que se repitieron mucho, pero
  no puede memorizar 30 000 versos: con citas nuevas suele adivinar un personaje plausible.
- **SFT + RAG** suele ser lo mejor: el formato aprendido + los hechos del corpus.
- ⚠️ El modelo solo «conoce» **esta edición**: en la de EMOTHE, *La vida es sueño* empieza con «Hipogrifo
  *triste*», no «violento». ¿Qué implica esto para la crítica textual y para usar LLM como fuente?

## 🖐️ Tu turno
1. Añade tus propias preguntas a `PREGUNTAS` (sobre el reparto, sobre citas, sobre la trama).
2. Cambia `rag_k` (1, 3, 6). ¿Mejoran las respuestas con más fragmentos o se confunde el modelo?
3. Busca con `indice.search("honor", k=5)` o con otra palabra clave: ¿en qué obras aparece?
4. **Para pensar:** ¿podríamos publicar este modelo ajustado? ¿Quién tiene derechos sobre una edición crítica?

## (Opcional) Ajuste de **estilo**: escribir «como Calderón»
Entrenamos unos minutos con el texto de 2–3 obras (sin preguntas) y pedimos al modelo que continúe un parlamento.
Pon `ESTILO = True` (mejor con GPU; carga un modelo nuevo).

In [ ]:
ESTILO = False
if ESTILO:
    from dhllm.finetune import make_chunks, finetune_on_texts, generate_hf
    textos_estilo = [cal.play_to_text(o) for o in obras[:3]]
    m2, tok2 = load_base_model(ajustes.base_model)
    tr, va = make_chunks(textos_estilo, tok2, block_size=256)
    m2 = prepare_model(m2, "lora", 16, 32)
    finetune_on_texts(m2, tr, va, {"batch_size": 8, "max_steps": 150, "learning_rate": 2e-4})
    print(generate_hf(m2, tok2, "Segismundo:\n    ¡Ay mísero de mí, ay infelice!", max_new_tokens=80))